# 1b — Matrix Transforms: Moving and Rotating a Scan

Scan matching asks: *what rigid motion (a shift plus a turn) lines up scan B
with scan A?* To even pose that question we need a compact way to apply a
rigid motion to a whole cloud of points. That tool is the **2D homogeneous
transformation matrix**.

**Learning goals**

1. Translate a point cloud by adding a vector.
2. Rotate a point cloud with a 2×2 rotation matrix.
3. Combine both into a single 3×3 matrix using homogeneous coordinates.
4. Compose and invert transforms, and see why order matters.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## A synthetic shape

An L-shaped corner is a good test object: unlike a single wall you can
tell when it has been rotated.

In [ ]:
def make_corner(num_points=40, length=4.0):
    """An L-shaped corner: one wall along +x, one along +y, meeting at the origin."""
    s = np.linspace(0, length, num_points)
    wall_x = np.column_stack((s, np.zeros_like(s)))
    wall_y = np.column_stack((np.zeros_like(s), s))
    return np.vstack((wall_x, wall_y))

corner = make_corner()

def plot_points(ax, pts, label, **kw):
    ax.scatter(pts[:, 0], pts[:, 1], s=12, label=label, **kw)
    ax.axis("equal"); ax.grid(True)

fig, ax = plt.subplots(figsize=(5, 5))
plot_points(ax, corner, "corner")
ax.legend(); plt.show()

## Translation

Shifting every point by the same vector $\mathbf{t} = (t_x, t_y)$ is just a
broadcasted addition: `points + t`.

In [ ]:
t = np.array([2.0, 1.0])
shifted = corner + t

fig, ax = plt.subplots(figsize=(5, 5))
plot_points(ax, corner, "original")
plot_points(ax, shifted, "translated by (2, 1)")
ax.legend(); plt.show()

## Rotation

Rotating a point counter-clockwise by $\theta$ about the origin is a matrix
multiplication:

$$
\begin{bmatrix} x' \\ y' \end{bmatrix} =
\begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}
\begin{bmatrix} x \\ y \end{bmatrix}
$$

Our points are stored as rows of an `(N, 2)` array, so instead of `R @ p`
for each column vector we write `points @ R.T` for the whole array.

In [ ]:
def rotation_matrix(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s],
                     [s,  c]])

theta = np.radians(30)
rotated = corner @ rotation_matrix(theta).T

fig, ax = plt.subplots(figsize=(5, 5))
plot_points(ax, corner, "original")
plot_points(ax, rotated, "rotated 30°")
ax.legend(); plt.show()

## Homogeneous coordinates: one matrix for both

Rotation is a matrix product but translation is an addition, so chaining
several motions gets messy. The trick is to append a `1` to each point,
$(x, y) \to (x, y, 1)$, so that translation *also* becomes a matrix
product:

$$
T =
\begin{bmatrix}
\cos\theta & -\sin\theta & t_x \\
\sin\theta & \cos\theta & t_y \\
0 & 0 & 1
\end{bmatrix},
\qquad
\begin{bmatrix} x' \\ y' \\ 1 \end{bmatrix} = T \begin{bmatrix} x \\ y \\ 1 \end{bmatrix}.
$$

Reading the matrix: the top-left 2×2 block rotates, the last column
translates. The rotation is applied **first**, then the translation.

In [ ]:
def transform_matrix(tx, ty, theta):
    """3x3 homogeneous transform: rotate by theta, then translate by (tx, ty)."""
    T = np.eye(3)
    T[:2, :2] = rotation_matrix(theta)
    T[:2, 2] = (tx, ty)
    return T

def apply_transform(T, points):
    """Apply a 3x3 homogeneous transform to an (N, 2) array of points."""
    homogeneous = np.column_stack((points, np.ones(len(points))))
    return (homogeneous @ T.T)[:, :2]

T = transform_matrix(2.0, 1.0, np.radians(30))
print(np.round(T, 3))

moved = apply_transform(T, corner)

# Same thing done by hand: rotate, then translate.
check = corner @ rotation_matrix(np.radians(30)).T + np.array([2.0, 1.0])
print("matches manual rotate-then-translate:", np.allclose(moved, check))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
plot_points(ax, corner, "original")
plot_points(ax, moved, "rotate 30°, then translate (2, 1)")
ax.legend(); plt.show()

## Composition, and why order matters

Two transforms chain by matrix multiplication. `T2 @ T1` means *apply `T1`
first, then `T2`* (the one nearest the points acts first). Rotations and
translations don't commute, so swapping the order gives a different result.

In [ ]:
R = transform_matrix(0, 0, np.radians(30))   # pure rotation
S = transform_matrix(2, 1, 0)                # pure translation

rotate_then_shift = apply_transform(S @ R, corner)
shift_then_rotate = apply_transform(R @ S, corner)

fig, ax = plt.subplots(figsize=(6, 6))
plot_points(ax, corner, "original")
plot_points(ax, rotate_then_shift, "rotate, then shift  (S @ R)")
plot_points(ax, shift_then_rotate, "shift, then rotate  (R @ S)")
ax.legend(); plt.show()

## Inverse transforms

If `T` moves the robot from pose A to pose B, then `inv(T)` moves it back.
NumPy can invert the 3×3 directly, but the closed form is cheap and worth
knowing: the inverse rotation is the transpose $R^\top$, and the inverse
translation is $-R^\top \mathbf{t}$.

In [ ]:
def invert_transform(T):
    R = T[:2, :2]
    t = T[:2, 2]
    T_inv = np.eye(3)
    T_inv[:2, :2] = R.T
    T_inv[:2, 2] = -R.T @ t
    return T_inv

T_inv = invert_transform(T)
print("closed form equals np.linalg.inv:", np.allclose(T_inv, np.linalg.inv(T)))
print("round trip recovers the corner:", np.allclose(apply_transform(T_inv, moved), corner))

## Connecting to the robot

Suppose the robot moves by `T` (it turns by θ and drives by `(tx, ty)`).
The *walls* did not move, so in the robot's new frame they appear to have
moved by `inv(T)`. That is exactly the convention used in the step-1
script: a robot motion of `(+dx, +dy)` shifts the observed points by
`(-dx, -dy)`.

A scan matcher works the other way round. Given the old and new scans, it
searches for the `T` that maps the new scan back onto the old one — and
that `T` **is** the robot's motion.

In [ ]:
robot_motion = transform_matrix(0.6, 0.3, np.radians(10))

scan_before = corner                                              # what the robot saw at t0
scan_after = apply_transform(invert_transform(robot_motion), corner)   # what it sees at t1

fig, ax = plt.subplots(figsize=(6, 6))
plot_points(ax, scan_before, "scan at t0")
plot_points(ax, scan_after, "scan at t1 (robot moved, walls appear to move back)")
plot_points(ax, apply_transform(robot_motion, scan_after), "t1 scan mapped back with the robot's motion", marker="x")
ax.legend(); plt.show()

## Summary

* Translation: `points + t`. Rotation: `points @ R.T`.
* A 3×3 homogeneous matrix does both at once; chain with `@`, the rightmost
  transform acts first.
* `invert_transform` undoes a motion; the robot moving by `T` makes the
  scene appear to move by `inv(T)`.

**Next:** Module 2 searches over `(tx, ty, θ)` to find the transform that
best aligns two scans — see
[2b — The cost function surface](2b_cost_function_surface.ipynb).